# 🐄 Boeuf Tracker — Google Colab

Lance l'UI complète du **Boeuf Tracker** (YOLOv11 + DINOv2 + Re-ID) sur le GPU gratuit de Colab.

**Pourquoi Colab ?**
- 🎮 **GPU T4 (16 GB)** ou **A100 (40 GB)** — bien plus de VRAM qu'une GTX 1660 Ti (6 GB)
- 🧠 Modèles "lourds" supportés : `yolo11l-seg` + `dinov2-base`
- 🌐 UI accessible depuis n'importe quel navigateur via un tunnel **Cloudflare** (gratuit, sans compte)
- 📊 **Moniteur live** intégré : voit en temps réel si la vidéo est traitée

**Prérequis**
1. Pousse ton repo sur GitHub (privé ou public)
2. Édite la cellule suivante : mets l'URL HTTPS dans `GIT_URL`
3. `Runtime` → `Change runtime type` → **T4 GPU** (ou A100)
4. Exécute les cellules dans l'ordre

À la fin, tu auras :
- Une URL publique `https://xxx.trycloudflare.com` pour l'UI
- Un **dashboard live** qui affiche FPS, frames, source, events, animaux visibles
- Le **log Flask** en temps réel (pour debug du switch de source)

In [ ]:
# =================================================================
# ⚙️  CONFIGURATION — modifie selon tes besoins
# =================================================================

# URL HTTPS de ton repo Git (obligatoire)
GIT_URL    = "https://github.com/ismaelgansonre/boeuf-tracker.git"   # <-- MODIFIE ICI
GIT_BRANCH = "fix/b36b1e9"
GIT_TOKEN  = ""        # PAT GitHub pour repo privé (optionnel)

# --- Modèles (plus gros = +précis mais +VRAM) ---
# T4 (16GB)  : yolo11l-seg + dinov2-base   ✅ confortable
# A100(40GB) : yolo11x-seg + dinov2-large  ✅ très précis
YOLO_MODEL  = "yolo11l-seg.pt"
DINO_MODEL  = "facebook/dinov2-base"

# --- Paramètres de détection ---
THRESHOLD   = 0.65     # Seuil cosine Re-ID (0.4 permissif → 0.8 strict)
CONF        = 0.4      # Confiance min YOLO (0.3 sensible → 0.6 strict)
IMGSZ       = 640      # Résolution YOLO (320 rapide → 1280 précis)
EMBED_EVERY = 10       # Re-embed tous les N frames (perf vs précision)

# --- Réseau ---
PORT = 5000

# --- Injection du token si repo privé ---
_repo_display = GIT_URL
if GIT_TOKEN and "@" not in GIT_URL and "github.com" in GIT_URL:
    GIT_URL = GIT_URL.replace("https://", f"https://x-access-token:{GIT_TOKEN}@")
    _repo_display = GIT_URL.replace(f"x-access-token:{GIT_TOKEN}@", "")

print(f"📦 Repo    : {_repo_display.replace('https://', '')}")
print(f"🤖 YOLO    : {YOLO_MODEL}")
print(f"🧠 DINOv2  : {DINO_MODEL}")
print(f"🔌 Port    : {PORT}")
print(f"⚙️  Settings: thr={THRESHOLD}  conf={CONF}  imgsz={IMGSZ}  embed_every={EMBED_EVERY}")

In [ ]:
import subprocess, sys

print("⏳ Installation des paquets système (ffmpeg)...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

print("⏳ Installation des dépendances Python...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.0.0",
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "transformers>=4.35.0",
    "opencv-python>=4.8.0",
    "numpy>=1.24.0",
    "Pillow>=10.0.0",
    "flask>=3.0.0",
    "werkzeug",
], check=True)

print("✅ Dépendances installées")

In [ ]:
import os, subprocess
from pathlib import Path

if "USER" in GIT_URL or not GIT_URL.strip():
    raise SystemExit(
        "❌ Configure GIT_URL dans la cellule précédente !\n"
        "   Exemple: GIT_URL = 'https://github.com/ismaelgansonre/boeuf-tracker.git'"
    )

repo_name = GIT_URL.rstrip("/").split("/")[-1].replace(".git", "")
if "@" in repo_name:
    repo_name = repo_name.split(":")[-1].split("/")[-1]

repo_path = Path("/content") / repo_name

if not repo_path.exists():
    print(f"⏳ Clonage du repo (branche {GIT_BRANCH})...")
    subprocess.run([
        "git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, str(repo_path)
    ], check=True)
    print("✅ Clone OK")
else:
    print(f"✅ Repo déjà présent: {repo_path}")
    print("   (supprime-le pour forcer un re-clone)")

os.chdir(repo_path)
print(f"📂 CWD = {os.getcwd()}")
print(f"📋 Contenu : {sorted(os.listdir('.'))[:15]}")

In [ ]:
import torch, subprocess
from pathlib import Path

HAS_GPU = torch.cuda.is_available()

if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU détecté : {gpu_name}")
    print(f"   VRAM: {vram_gb:.1f} GB")
    if "T4" in gpu_name.upper():
        if "l-seg" in YOLO_MODEL or "large" in DINO_MODEL.lower():
            print("⚠️  T4 détecté. Si OOM en cours de route, baisse vers yolo11s-seg + dinov2-small.")
else:
    print("⚠️  AUCUN GPU DÉTECTÉ — le serveur tournera sur CPU (très lent, ~2-5 FPS)")
    print()
    print("=" * 70)
    print("   Pour activer le GPU T4 (gratuit dans Colab) :")
    print("   1. Menu → Runtime → Change runtime type")
    print("   2. Hardware accelerator → T4 GPU")
    print("   3. Save")
    print("   ⚠️  Changer le runtime EFFACE toutes les variables !")
    print("      → Il faudra ré-exécuter TOUTES les cellules depuis le début.")
    print("=" * 70)
    print()
    print("Tu peux quand même continuer sur CPU (cell suivante),")
    print("mais ça sera 5-10× plus lent.")
    print()

# Téléchargement des poids YOLO si absents
yolo_path = Path(YOLO_MODEL)
if not yolo_path.exists():
    print(f"⏳ Téléchargement de {YOLO_MODEL} (~50-100 MB)...")
    tag = "v8.3.0"
    url = f"https://github.com/ultralytics/assets/releases/download/{tag}/{YOLO_MODEL}"
    subprocess.run(["wget", "-q", "--show-progress", url], check=True)
    print("✅ Poids YOLO téléchargés")

print("\n📦 Modèles .pt dans le dossier :")
for p in sorted(Path(".").glob("*.pt")):
    print(f"   - {p.name:25s}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")


In [ ]:
import subprocess, time, os, urllib.request, urllib.error
from pathlib import Path

# Nettoyage des runs précédents
subprocess.run(["pkill", "-f", "python app.py"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "cloudflared"],    stderr=subprocess.DEVNULL)
time.sleep(2)

flask_cmd = [
    "python", "app.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--device", "cuda:0" if HAS_GPU else "cpu",
    "--yolo-model", YOLO_MODEL,
    "--dino-model", DINO_MODEL,
    "--threshold", str(THRESHOLD),
    "--conf",      str(CONF),
    "--imgsz",     str(IMGSZ),
    "--embed-every", str(EMBED_EVERY),
]

log_path = Path("/tmp/flask.log")
log_path.unlink(missing_ok=True)
log_f = open(log_path, "wb", 0)

print("⏳ Démarrage du serveur Flask...")
print("   (chargement YOLO + DINOv2 + DB → ~30-60s)")
print()

flask_proc = subprocess.Popen(
    flask_cmd,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

ready = False
for i in range(120):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats", timeout=1).read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError):
        if flask_proc.poll() is not None:
            print("❌ Le serveur a crashé. Dernières lignes du log :")
            subprocess.run(["tail", "-40", "/tmp/flask.log"])
            raise SystemExit(1)
        if i % 5 == 0:
            print(f"   ... chargement ({i*2}s)")

if not ready:
    print("❌ Timeout (4 min).")
    subprocess.run(["tail", "-60", "/tmp/flask.log"])
    raise SystemExit(1)

print(f"✅ Serveur Flask prêt sur http://127.0.0.1:{PORT}")
print(f"   PID Flask = {flask_proc.pid}")

import json as _j
try:
    stats = _j.loads(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats").read())
    device_str = stats.get('device', '?')
    device_icon = "🎮" if device_str.startswith('cuda') else "🐌"
    print(f"   {device_icon} Device actif : {device_str}")
    if not device_str.startswith('cuda'):
        print("   ⚠️  CPU détecté → active T4 GPU dans Runtime pour de la perf")
    print(f"   YOLO chargé  : {stats.get('current', {}).get('yolo_model')}")
except Exception as e:
    print(f"   (stats non lisibles: {e})")

In [ ]:
import subprocess, re, time, os, urllib.request, json
from pathlib import Path
from IPython.display import clear_output

# === Installation cloudflared ===
if not Path("/usr/local/bin/cloudflared").exists() and not Path("/usr/bin/cloudflared").exists():
    print("⏳ Installation de cloudflared...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        "-O", "/tmp/cloudflared.deb",
    ], check=True)
    subprocess.run(["dpkg", "-i", "/tmp/cloudflared.deb"], check=True)

# === Démarrage tunnel ===
tunnel_log = Path("/tmp/tunnel.log")
tunnel_log.unlink(missing_ok=True)
tunnel_f = open(tunnel_log, "wb", 0)

print("⏳ Création du tunnel Cloudflare...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

url = None
for i in range(90):
    time.sleep(2)
    content = tunnel_log.read_text(errors="ignore")
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', content)
    if m:
        url = m.group(1)
        break
    if tunnel_proc.poll() is not None:
        print("❌ Tunnel crashé. Log :")
        print(content[-1500:])
        raise SystemExit(1)

if not url:
    print("❌ Pas d'URL publique après 3 min.")
    print(tunnel_log.read_text(errors="ignore")[-2000:])
    raise SystemExit(1)

print()
print("=" * 70)
print(f"  🌐  UI ACCESSIBLE À :")
print(f"      {url}")
print("=" * 70)
print()
print("📊 Démarrage du MONITEUR LIVE (rafraîchissement toutes les 3s)...")
print("   → Upload une vidéo via l'UI et regarde le moniteur")
print("   → Ctrl+C pour ARRÊTER le monitoring (serveur + tunnel restent UP)")
print()


# =====================================================================
# === Helpers pour le moniteur ===
# =====================================================================

def get_stats():
    """GET /api/stats avec timeout court."""
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats", timeout=2) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"_error": str(e)}


def get_videos():
    """GET /api/videos — liste des vidéos détectées par l'app."""
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/videos", timeout=2) as r:
            return json.loads(r.read()).get("videos", [])
    except Exception:
        return []


def get_diag():
    """GET /api/diag — diagnostic complet."""
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/diag", timeout=2) as r:
            return json.loads(r.read())
    except Exception:
        return {}


def tail_log(n=12):
    """Lit les N dernières lignes du log Flask (non-bloquant)."""
    try:
        with open("/tmp/flask.log", "rb") as f:
            f.seek(0, 2)               # fin du fichier
            size = f.tell()
            f.seek(max(0, size - 8192))
            data = f.read().decode(errors="ignore")
            lines = data.splitlines()
            return lines[-n:]
    except Exception:
        return []


def send_source(path):
    """POST /api/source/file — force le switch vers un fichier (debug)."""
    try:
        req = urllib.request.Request(
            f"http://127.0.0.1:{PORT}/api/source/file",
            data=json.dumps({"path": path}).encode(),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=5) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"ok": False, "error": str(e)}


# =====================================================================
# === Boucle du moniteur live ===
# =====================================================================

prev_fc = -1
prev_src = None

try:
    while True:
        clear_output(wait=True)

        # ---------- BANNIÈRE URL ----------
        print("=" * 70)
        print(f"  🌐  UI LIVE : {url}")
        print("=" * 70)

        # ---------- ÉTAT SERVEUR ----------
        stats = get_stats()
        diag  = get_diag()

        if "_error" in stats:
            print(f"\n❌ /api/stats inaccessible : {stats['_error']}")
            print(f"   → Le serveur Flask est peut-être mort. Regarde le log ci-dessous.")
        else:
            fps     = stats.get("fps", 0)
            fc      = stats.get("frame_count", 0)
            src     = stats.get("source_label", "?")
            dev     = stats.get("device", "?")
            events  = stats.get("events", [])
            cur     = stats.get("current", {})
            active  = stats.get("active", [])
            behavior = stats.get("behavior", [])

            # Indicateur de progression
            if fc == 0 and not events:
                status = "🟡 INITIALISATION — aucune frame traitée"
            elif fc == 0 and events:
                status = "🟠 0 FRAME — des events sont loggés mais pas de frame"
            elif fc > prev_fc:
                status = f"🟢 TRAITEMENT OK (Δ +{fc - prev_fc})"
            elif fc == prev_fc and fc > 0:
                status = "🟠 STAGNANT — frames ne progressent plus"
            else:
                status = "🔴 BLOQUÉ"

            src_changed = "  🔄 SOURCE CHANGÉE !" if src != prev_src else ""

            print(f"\n{status}{src_changed}")
            print(f"   📂 Source      : {src}")
            print(f"   🎮 Device      : {dev}")
            print(f"   🎬 FPS         : {fps:.1f}")
            print(f"   🖼️  Frames      : {fc}")
            print(f"   🤖 YOLO        : {cur.get('yolo_model', '?')}")
            print(f"   📐 imgsz       : {cur.get('imgsz', '?')}")
            print(f"   🎚️  threshold   : {cur.get('threshold', '?')}")
            print(f"   🎯 conf        : {cur.get('conf', '?')}")

            # ---------- ÉVÉNEMENTS ----------
            if events:
                print(f"\n📝 Derniers événements (du + récent au + ancien) :")
                for e in events[:10]:
                    print(f"   • {e}")
            else:
                print(f"\n📝 Aucun événement pour l'instant")
                print(f"   💡 Upload une vidéo via l'UI → elle apparaîtra ici en 'NEW' ou 'MATCH'")

            # ---------- ANIMAUX VISIBLES ----------
            if active:
                print(f"\n🐄 Animaux visibles ({len(active)}) :")
                for a in active[:8]:
                    print(f"   • {a.get('name', '?'):15s}  conf={a.get('conf', 0):.2f}  track_id={a.get('track_id', '?')}")
            else:
                print(f"\n🐄 Aucun animal visible actuellement")

            # ---------- COMPORTEMENTS ----------
            if behavior:
                print(f"\n🏃 Comportements :")
                for b in behavior[:5]:
                    print(f"   • {b.get('name', '?'):15s} → {b.get('action', '?')} (vitesse={b.get('speed', 0):.1f})")

        # ---------- DIAGNOSTIC ----------
        desired_src = diag.get("state", {}).get("desired_source", None)
        cur_src     = diag.get("state", {}).get("current_source_path", None)
        if desired_src:
            print(f"\n🔄 Switch demandé vers : {desired_src}")
        if cur_src and cur_src != 0:
            print(f"✅ Source actuelle (interne) : {cur_src}")

        # ---------- VIDÉOS DISPONIBLES ----------
        videos = get_videos()
        if videos:
            print(f"\n🎬 Vidéos détectées par l'app ({len(videos)}) :")
            for v in videos[:5]:
                print(f"   • [{v.get('source', '?'):8s}] {v.get('name', '?')}  ({v.get('size_mb', 0):.1f} MB)")

        # ---------- LOG FLASK (dernières lignes) ----------
        log_lines = tail_log(12)
        if log_lines:
            print(f"\n📜 Log Flask (12 dernières lignes — couleurs perdues) :")
            for line in log_lines:
                if line.strip():
                    print(f"   {line[:200]}")

        # ---------- TEST HELPER ----------
        print(f"\n" + "─" * 70)
        print(f"💡 TESTS RAPIDES (à exécuter dans une autre cellule si besoin) :")
        print(f"   # Forcer le switch vers une vidéo :")
        print(f"   send_source('/content/boeuf-tracker/MA_VIDEO.mp4')")
        print(f"   # Voir l'état complet :")
        print(f"   get_stats()")
        print(f"   get_videos()")
        print("─" * 70)
        print(f"⏳ Prochain refresh dans 3s ... (Ctrl+C pour stopper le monitoring)")

        prev_fc  = fc if 'fc' in dir() else 0
        prev_src = src if 'src' in dir() else None

        time.sleep(3)

except KeyboardInterrupt:
    clear_output(wait=True)
    print()
    print("=" * 70)
    print("🛑  MONITEUR ARRÊTÉ")
    print("=" * 70)
    print(f"✅ Serveur Flask    : toujours UP  (PID via cell 6)")
    print(f"✅ Tunnel Cloudflare : toujours UP")
    print(f"🌐 URL              : {url}")
    print()
    print("📌 Pour relancer le monitoring : ré-exécute cette cellule.")
    print("📌 Pour forcer un switch de source manuellement :")
    print(f"   send_source('/content/boeuf-tracker/172777-847860598_medium.mp4')")
    print()
    print("=" * 70)

In [ ]:
# OPTIONNEL: télécharge une vidéo d'exemple pour tester rapidement.
# Sinon, utilise le bouton 'Upload' dans l'UI (upload direct navigateur → Colab).

SAMPLE_URL = ""   # ← colle ici une URL directe vers un .mp4 de bovins
if SAMPLE_URL:
    print(f"⏳ Téléchargement de la vidéo d'exemple...")
    subprocess.run(["wget", "-q", "--show-progress", SAMPLE_URL, "-O", "sample_cattle.mp4"], check=True)
    size_mb = Path("sample_cattle.mp4").stat().st_size / 1024 / 1024
    print(f"✅ Vidéo téléchargée: sample_cattle.mp4 ({size_mb:.1f} MB)")
    print(f"   → Sélectionne-la dans le menu déroulant 'Source' de l'UI.")
else:
    print("💡 Pas de SAMPLE_URL configuré.")
    print("   → Upload ta vidéo via le bouton 'Upload' dans l'UI.")
    print("   → Ou mets une URL directe ci-dessus et ré-exécute.")

## 🎉 C'est en ligne !

Ouvre l'URL affichée par le moniteur dans ton navigateur. Tu devrais voir :
- Le **flux MJPEG** en temps réel avec les silhouettes des bovins
- Les **sliders** pour ajuster seuil Re-ID, confiance, imgsz à chaud
- Le **journal** des événements (`NEW`, `MATCH`, `LOOP`, etc.)
- La **liste des animaux** détectés avec leur couleur stable

### 📺 Pendant que l'UI tourne

Le **moniteur live** (cellule précédente) t'affiche en temps réel :
- 🟢/🟠/🔴 état du processing (frames qui avancent ou bloquées)
- 📝 tous les events que le serveur loggue (`NEW Boeuf_001`, `MATCH Marguerite`, etc.)
- 📜 les 12 dernières lignes du log Flask (erreurs, switches de source, OOM…)
- 🎬 les vidéos détectées par l'app

Tu peux **uploader une vidéo dans l'UI** pendant que le moniteur tourne et voir immédiatement les events apparaître.

### 🛑 Pour arrêter le monitoring (sans tuer le serveur)
- Bouton **⏹ Stop** de la cellule, OU Ctrl+C
- Le serveur Flask + le tunnel continuent à tourner

### 🛑 Pour tout arrêter
- Ferme l'onglet Colab, ou `Runtime` → `Manage sessions` → Terminate

### 🔧 Troubleshooting grâce au moniteur

| Symptôme dans le moniteur | Cause probable | Solution |
|---|---|---|
| `🟡 INITIALISATION` + log vide | YOLO charge encore | Attendre 30-60s |
| `🟠 0 FRAME — events loggés` | Vidéo lue mais pas de bovin détecté | Monte YOLO ou descends `conf` |
| `🟠 STAGNANT` | Vidéo finie / bloquée sur webcam | Sélectionne une autre source dans l'UI |
| `🔄 SOURCE CHANGÉE !` mais reste sur webcam | Fichier vidéo illisible par OpenCV | Vérifie le codec (H.264 conseillé) |
| Events `NEW` mais aucun `MATCH` | DB vide (normal au 1er run) | Normal ! Les bovins seront nommés Boeuf_001... |
| Log montre `OOM CUDA` | VRAM insuffisante | Baisse `YOLO_MODEL` à `yolo11s-seg.pt` |
| Log montre `ERREUR ouverture` | Fichier vidéo introuvable | Vérifie le chemin, ou utilise Upload UI |

### 💡 Forcer un switch de source depuis le notebook

Si l'UI ne switch pas correctement, exécute dans une cellule :
```python
send_source("/content/boeuf-tracker/uploads/172777-847860598_medium.mp4")
```